<h1 align="center">Rwanda Maize–Urea Trial</h1>
<h2 align="center">5. Data Enrichment: Rainfall & Soil</h2>

**Goal:** fill two gaps the brief highlights:
1. **Weather:** there is no weather data. This notebook adds `rain_plant_silking`, total rainfall from planting to silking, from **CHIRPS v2.0** daily rainfall (~5 km cells). It is downloaded from NOAA's ERDDAP server and needs nothing beyond pandas.
2. **Soil:** `tot_N`, `org_C` and `sand` are completely empty. They are filled from **SoilGrids 2.0** (ISRIC, ~250 m cells).

**Running this notebook downloads the data** (internet access required). Every response is cached in `01_Data/external/cache/`, so:
* a second run takes seconds;
* if a run is interrupted, re-running continues where it stopped.

Approximate first-run times:
* **CHIRPS:** a few minutes, one small request per 5 km cell.
* **SoilGrids:** about 45 minutes. ISRIC's REST API allows ~5 requests per minute and the trial has ~220 distinct locations. For a faster route, install `rasterio` and set `SOIL_SOURCE = "vrt"` to read the global rasters directly.

The same code can also be run from a terminal: `python scripts/fetch_chirps_rainfall.py` and `python scripts/fetch_soilgrids.py`.

> **Data note.** The trial data is confidential and is not in this repository. The outputs shown come from the **synthetic sample** in `01_Data/sample/`, which has the same structure and data-quality problems as the real data. The numbers illustrate the method; they are not Evergreen's actual results.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "02_Notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from maize import config, data

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
MODE = config.data_mode()
OUT = config.output_paths(MODE)
print("Data source:", "confidential trial data (01_Data/raw)" 
      if MODE == "raw" else "synthetic sample (01_Data/sample)")

sys.path.insert(0, str(ROOT / "scripts"))
import fetch_chirps_rainfall
import fetch_soilgrids

Data source: synthetic sample (01_Data/sample)


## 1. What needs to be fetched

In [2]:
src = data.load_sources(MODE)
fields = src["field"]
pd.Series({
    "Field-seasons": len(fields),
    "Distinct SoilGrids locations (~100 m)": fields[["lat", "long"]].round(3).drop_duplicates().shape[0],
    "Soil variables 100% empty": ", ".join(c for c in ["tot_N", "org_C", "sand"] if src["soil"][c].isna().all()),
    "Plots needing a rainfall window": len(src["plot"]),
    "Plots without a silking date (window undefined)": src["plot"]["silking_date"].isna().sum(),
}, name="").to_frame()

,
Field-seasons,397
Distinct SoilGrids locations (~100 m),397
Soil variables 100% empty,"tot_N, org_C, sand"
Plots needing a rainfall window,1590
Plots without a silking date (window undefined),9


**Method choices**
* **Rainfall window.** Rainfall is summed daily from planting to silking, inclusive, for the CHIRPS cell containing each field. Plots without a silking date, or with dates out of order, are left missing.
* **SoilGrids depth.** Values are averaged over 0–30 cm, weighted by layer thickness (0–5, 5–15, 15–30 cm), which matches the maize rooting zone where fertiliser is applied. Units: N and SOC in g/kg, sand and clay in %, pH in water.
* **Modelled vs measured.** SoilGrids values go in separate `sg_*` columns. The measured pH is never overwritten.
* **Brief deliverable 5.** `01_Data/external/Soil_PL_filled.csv` is the soil table for every plot, with `tot_N` and `org_C` (%) and `sand` (%) filled, plus a source column.

## 2. Download

The downloads are switched **off** for the synthetic sample: its field coordinates are made up, so rainfall and soil fetched for them would mean nothing. Set both flags to `True` when running with the real trial data in `01_Data/raw/`.

In [3]:
# Off for the synthetic sample (made-up coordinates); set to True for the real trial data
FETCH_RAINFALL = False
FETCH_SOIL = False
SOIL_SOURCE = "rest"   # "rest" (no extra packages, ~45 min first time) or "vrt" (needs rasterio, faster)

In [4]:
if FETCH_RAINFALL:
    try:
        rain = fetch_chirps_rainfall.run(source="erddap", mode=MODE)
        display(rain["rain_plant_silking"].describe().to_frame("rain_plant_silking (mm)").T)
    except Exception as e:
        print("CHIRPS download failed:", e)
        print("Check the internet connection, or try fetch_chirps_rainfall.run(source='iridl') "
              "(needs: pip install xarray netCDF4).")

In [5]:
if FETCH_SOIL:
    try:
        soilgrids = fetch_soilgrids.run(source=SOIL_SOURCE, mode=MODE)
        display(soilgrids.filter(like="sg_").describe().T)
    except Exception as e:
        print("SoilGrids download failed:", e)
        print("Progress so far is cached - re-run this cell to continue, "
              "or set SOIL_SOURCE = 'vrt' (needs: pip install rasterio).")

## 3. Status of the enrichment files

In [6]:
external = data.load_external()
pd.Series({
    "CHIRPS rainfall (chirps_rain_plant_silking.csv)": "rainfall" in external,
    "SoilGrids soil (soilgrids_soil.csv)": "soilgrids" in external,
    "Filled soil table (Soil_PL_filled.csv)": (config.EXTERNAL_DIR / "Soil_PL_filled.csv").exists(),
}, name="available").to_frame()

,available
CHIRPS rainfall (chirps_rain_plant_silking.csv),False
SoilGrids soil (soilgrids_soil.csv),False
Filled soil table (Soil_PL_filled.csv),False


## 4. Explore the added variables

In [7]:
_, combined = data.build_combined(MODE)
added = [c for c in combined.columns if c.startswith(("sg_", "rain_"))]
if not added:
    print("No enrichment data yet - check the messages in section 2.")
else:
    ok = combined[~combined["flag_implausible_yield"]]
    print("Coverage (share of plots with a value):")
    display(ok[added].notna().mean().to_frame("coverage").T)
    display(ok.groupby("meg_env")[added].mean())
    display(ok[added + ["yield_tha"]].corr()["yield_tha"].drop("yield_tha").to_frame("correlation with yield"))

No enrichment data yet - check the messages in section 2.


In [8]:
if "rain_plant_silking" in combined:
    import matplotlib.pyplot as plt
    plotting_df = ok.dropna(subset=["rain_plant_silking"])
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, s in enumerate(config.SEASONS):
        d = plotting_df[plotting_df["season"] == s]
        ax.scatter(d["rain_plant_silking"], d["yield_tha"], s=10, alpha=.5, color=config.SEASON_COLORS[s], label=s)
    ax.set_xlabel("Rainfall, planting to silking (mm)")
    ax.set_ylabel("Yield (t/ha)")
    ax.set_title("Yield vs in-season rainfall", loc="left", fontweight="bold")
    ax.legend(title="Season", frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(OUT.figures / "yield_vs_rainfall.png", dpi=160, bbox_inches="tight")
    plt.show()

## What happens next

Once the files exist, every later step uses them automatically:
* **Notebook 06** adds `rain_plant_silking` and the `sg_*` variables to the driver-importance ranking.
* **`python run_pipeline.py`** rebuilds the combined dataset, the tables and the figures with the new variables.
* **The audit log** marks the soil and weather items as resolved.

**Next:** notebook 06, yield prediction and drivers.